# 공지를 맡기는 첫 모델 호출과 프롬프트 쓰기

첫 모임 공지를 받은 신입 회원은 어디로 가야 할까요? 진행자는 몇 시에 도착해야 할까요? 같은 공지로 두 사람에게 다른 안내를 써 봅니다. 노트북의 응답 칸에는 2026년 11월 26일 실제로 받은 답을 실었습니다. 내가 요청한 뒤에는 새 응답이 그 자리에 나타납니다.

## LLM과 트랜스포머를 이해하고 첫 요청 보내기

### LLM은 어떻게 문장을 쓸까요?

“비가 와서 우산을 ___.” 빈칸에 ‘폈다’도, ‘챙겼다’도 자연스럽습니다. **대규모 언어 모델(LLM, Large Language Model)**은 많은 글로 학습하면서 이런 문맥에서 어떤 표현이 이어질 가능성이 큰지 배웁니다. 이 수업에서 다루는 생성형 언어 모델은 많은 글로 미리 배우는 단계(사전 학습)에서 글의 앞부분으로 다음 조각을 예측하고, 원문에 실제로 나온 조각의 확률이 높아지도록 모델 안의 수많은 수치(매개변수)를 조정합니다.

모델이 다루는 글의 조각을 **토큰**이라고 합니다. 한 단어가 한 토큰이 될 수도 있고 여러 토큰으로 나뉠 수도 있습니다. 모델은 답을 쓸 때 다음 토큰 후보마다 확률을 계산해 하나를 고릅니다. 선택한 토큰까지 문맥에 넣고 다음 토큰을 고르는 일을 반복합니다. 같은 질문에도 ‘우산을 폈다’와 ‘우산을 펼쳤다’처럼 표현이 달라질 수 있는 이유입니다.

이 수업에서 공지를 입력하면 모델은 학습해 둔 매개변수를 이용해 **지금 보낸 공지와 요청에 이어질 답**을 만듭니다. 공지의 장소를 201호에서 305호로 고쳐 보내는 것은 이번 요청에 쓸 근거를 바꾸는 일입니다.

### 트랜스포머는 앞 문장을 어떻게 활용할까요?

트랜스포머(Transformer)는 오늘날 많은 LLM이 쓰는 신경망 구조입니다. 토큰을 숫자들의 묶음으로 바꾸고 순서 정보도 반영합니다. 그런 다음 여러 층에서 다른 위치의 정보를 섞어 각 위치가 문맥에서 어떤 뜻을 갖는지 표현합니다. 이때 **어떤 위치의 정보를 얼마나 반영할지** 계산하는 방법이 셀프 어텐션(self-attention)입니다.

“민지가 배를 깎았다. 여기서 배는 무엇일까?”를 읽으면 ‘깎았다’가 과일을 떠올리게 합니다. “민지가 배를 타고 섬에 갔다”로 바꾸면 ‘타고’와 ‘섬’이 탈것이라는 단서가 됩니다. 답을 생성하는 위치에서는 질문 전체가 앞에 놓여 있으므로, 같은 ‘배’가 들어 있어도 함께 들어온 단서에 따라 다음 답의 후보가 달라질 수 있습니다. 이 예는 사람이 읽을 수 있는 단서를 보여 줍니다. 모델 내부에서 측정한 어텐션 점수는 아닙니다.

공지에서도 마찬가지입니다. “신입 회원은 이어폰을 챙기고, 진행자는 20분 먼저 오세요”라는 같은 원문에 **누구에게 보낼 글인지** 덧붙여 보세요. ‘신입 회원에게’라는 요청에는 이어폰이, ‘진행자에게’라는 요청에는 도착 시간이 유용한 단서입니다. 다음 실습에서 실제 응답이 그 차이를 살렸는지 확인합니다.

### 공지에 없는 답도 자연스럽게 들릴까요?

공지에는 “금요일 오후 4시에 첫 모임을 합니다”라고만 적혀 있습니다. 모델이 “금요일 오후 4시, 가온관 201호에서 만나요”라고 썼다면 어느 부분을 그대로 보내도 될까요? 직접 원문에서 확인할 수 있는 부분에만 밑줄을 그어 보세요.

<details><summary>판단 예시 보기</summary><p>요일과 시간은 공지에서 확인할 수 있습니다. 장소는 알 수 없습니다. ‘가온관 201호’가 문장에 자연스럽게 이어져도 이 모임의 장소라는 근거는 없습니다.</p><p>그래서 첫 요청을 쓸 때는 원문을 함께 주고 “공지에 장소가 없으면 ‘미기재’라고 써 줘”처럼 빈 정보의 처리 방법을 정합니다. 답을 받은 뒤에는 원문에 있는 장소도 제대로 옮겼는지 확인합니다.</p></details>

### 노트북에서 호출 준비하기

수업용 Python 3 커널과 OpenAI API를 사용할 수 있는 키가 필요합니다. SDK는 Python 코드에서 API에 요청을 보내고 답을 받게 해 주는 패키지입니다. 처음 사용하는 커널에서는 새 코드 셀에 아래 설치 명령을 한 번 실행하고 커널을 다시 시작하세요. 이미 설치했다면 바로 준비 셀로 갑니다.

```python
%pip install openai==3.35.1
```

수업 환경에 OPENAI_API_KEY가 설정되어 있으면 자동으로 읽습니다. 개인 노트북에서는 아래 셀의 숨김 입력란에 키를 붙여 넣고 Enter를 누르세요. 빈 셀에 키를 그대로 적어 저장하면 파일을 받은 사람에게도 보입니다. 준비 셀만으로 모델을 호출하지는 않습니다.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키 입력: ")  # 입력한 글자는 표시되지 않습니다.
client = OpenAI()
model = "gpt-4.1-mini"  # 수업에서 사용할 수 있는 모델 이름을 지정합니다.

### 지시문과 원문을 따로 보내기

프롬프트는 모델에게 건네는 요청입니다. 이 예제에서는 instructions에 할 일을 적고, input에 실제로 읽을 공지를 넣습니다. 말투나 독자만 바꾸고 싶을 때는 공지를 다시 쓸 필요가 없습니다. 다음 셀을 한 번 실행하고 답이 돌아올 때까지 기다리세요. 모델을 부르는 셀을 다시 실행하면 새 요청과 사용량이 발생합니다.

In [ ]:
notice = '5월 21일 금요일 오후 4시에 가온관 201호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.'

instruction = '처음 오는 회원이 모임을 찾아올 수 있도록 날짜, 시간, 장소를 한 문장으로 안내해 주세요.'
response = client.responses.create(
    model=model,
    instructions=instruction,  # 모델이 해야 할 일
    input=notice,               # 이번에 읽을 공지 원문
    store=False,
)
print(response.output_text)  # 답변 본문만 읽습니다.

소리채집 동아리 첫 모임은 5월 21일 금요일 오후 4시에 가온관 201호에서 열리니, 처음 오시는 회원은 꼭 참석해 주세요.


실제 답의 ‘5월 21일 금요일 오후 4시’와 ‘가온관 201호’를 원문에서 찾아보세요. 날짜와 장소는 맞습니다. 끝의 ‘꼭 참석해 주세요’는 모델이 덧붙인 권유입니다. 참석을 독려할 권한이 없는 안내라면 지시문에 “참석 권유 문구는 생략하세요”를 더해 시험해 보세요. <mark>답을 읽을 때는 빠진 사실과 새로 생긴 주장을 따로 찾으세요.</mark>

In [ ]:
print(response.model)  # 방금 답한 모델
print(response.usage.input_tokens, response.usage.output_tokens)  # 보낸 양과 받은 양

gpt-4.1-mini-2025-04-14
108 45


108과 45는 각각 입력과 출력의 토큰 수입니다. 토큰은 모델이 글을 처리할 때 사용하는 단위입니다. 이 셀은 앞에서 받은 응답을 읽으므로 새 모델 요청을 보내지 않습니다. 처음 요청에서 AuthenticationError가 나오면 키 입력을 확인하세요. 모델 접근 오류라면 수업 계정이 사용할 수 있는 모델명을 강사와 확인합니다.

### 실습 · 장소가 바뀐 공지에 같은 요청 보내기

아래 공지에서 201호가 305호로 바뀌었습니다. 앞 셀의 지시는 유지하고 두 번째 응답을 받는 코드를 직접 작성하세요. 활동지에 답에서 읽은 날짜·시간·호수를 적고 첫 답과 비교합니다.

In [ ]:
new_notice = '5월 21일 금요일 오후 4시에 가온관 305호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.'
# 같은 instruction과 새 공지 new_notice를 넣어 모델을 호출하세요.

<details><summary>모범 프롬프트와 응답 살펴보기</summary><pre><code>updated = client.responses.create(
    model=model, instructions=instruction, input=new_notice, store=False
)
print(updated.output_text)</code></pre><p>이 지시로 받은 실제 응답</p><pre>소리채집 동아리 첫 모임은 5월 21일 금요일 오후 4시에 가온관 305호에서 열리니, 처음 오시는 회원께서는 이어폰을 꼭 챙겨 와 주세요.</pre><p>두 번째 응답은 305호를 안내합니다. 원문에서 바뀌지 않은 날짜와 시간도 유지됐습니다. 응답마다 곁들인 안내는 달라질 수 있습니다.</p></details>

## 누가 무엇을 해야 하는지 프롬프트에 담기

### “정리해 줘”라고 하면 무엇을 받을까요?

하나의 원문을 세 번 보냅니다. 처음에는 읽을 사람을 지정하지 않고, 다음에는 신입 회원, 마지막에는 진행자를 지정합니다. 호출 코드에서 같은 부분은 ask 함수에 모았습니다. 첫 인자는 instructions로, 두 번째 인자는 input으로 보냅니다. 함수가 돌려주는 값은 화면에 표시할 답변 텍스트입니다.

In [ ]:
def ask(instructions, source):
    reply = client.responses.create(
        model=model, instructions=instructions, input=source, store=False
    )
    return reply.output_text

print(ask("이 공지를 정리해 주세요.", notice))

공지 요약입니다:

- 일시: 5월 21일 금요일 오후 4시  
- 장소: 가온관 201호  
- 대상: 소리채집 동아리 신입 회원  
- 준비물: 신입 회원은 이어폰 지참  
- 진행자: 모임 20분 전에 도착하여 녹음기를 각 책상에 하나씩 배치할 것


일시·장소·준비물·진행자 업무가 목록으로 나왔습니다. 회의 자료로 훑기에는 편하지만, 신입 회원에게 그대로 보내면 자기와 관계없는 녹음기 배치까지 읽어야 합니다. 문장을 줄이기 전에 **누가 답을 읽고 다음에 무엇을 할지** 정해 봅시다.

In [ ]:
member_prompt = '소리채집 동아리의 신입 회원에게 지금 해야 할 행동을 한 문장으로 알려 주세요. 공지에 있는 사실만 사용하세요.'
print(ask(member_prompt, notice))

신입 회원은 5월 21일 금요일 오후 4시에 가온관 201호에서 열리는 첫 모임에 이어폰을 챙겨 참석하세요.


신입 회원이 할 일인 이어폰 준비가 나타났습니다. “신입 회원에게”는 독자를 정하고, “지금 해야 할 행동”은 고를 내용을 정하며, “한 문장”은 결과의 길이를 정합니다. 안내가 짧아도 준비물이 빠졌다면 목적을 달성하기 어렵습니다.

### 실습 · 진행자에게 보내는 한 문장

같은 공지로 진행자의 문자 안내를 만드세요. 신입 회원용 지시를 복사해 독자만 고칩니다. 직접 호출한 뒤 앞 답에 없던 준비 행동을 짚으세요. 이어폰 준비와 기기 배치가 섞였을 때는 어떤 지시를 더하면 좋을지 적어 보세요.

In [ ]:
# member_prompt를 참고해 진행자용 지시를 쓰고 ask에 notice와 함께 보내세요.

<details><summary>모범 프롬프트와 응답 살펴보기</summary><pre><code>organizer_prompt = &#x27;소리채집 동아리의 진행자에게 지금 해야 할 행동을 한 문장으로 알려 주세요. 공지에 있는 사실만 사용하세요.&#x27;
print(ask(organizer_prompt, notice))</code></pre><p>이 지시로 받은 실제 응답</p><pre>모임 20분 전인 오후 3시 40분에 가온관 201호에 도착하여 녹음기를 책상마다 한 대씩 놓으세요.</pre><p>3시 40분은 오후 4시에서 20분을 뺀 시각입니다. 책상마다 한 대씩 배치하라는 조건도 보존했습니다. 응답이 다른 역할의 일까지 담으면 “진행자가 직접 맡은 행동만 포함하세요”를 더해 다시 시험해 봅니다.</p></details>

### 내 프롬프트를 고칠 때 무엇을 남길까요?

같은 원문과 모델에서 한 번에 조건 하나를 바꾸면 어느 지시를 시험했는지 설명할 수 있습니다. 단 한 번의 답으로 효과를 단정하기는 어렵습니다. 다른 공지에서도 같은 문제가 되풀이되는지 확인하세요.

| 받은 답에서 발견한 문제 | 다음 요청에 넣어 볼 말 | 다시 읽을 부분 |
|---|---|---|
| 일정까지 빠질 만큼 짧아짐 | “일시·장소를 남긴 한 문장으로 써 주세요.” | 날짜와 호수가 함께 남았나요? |
| 너무 딱딱한 안내문 | “신입 회원 단체 채팅방에 보낼 존댓말로 써 주세요.” | 바로 보낼 수 있는 말투인가요? |
| 누가 할 일인지 섞임 | “진행자가 직접 준비할 일만 시간 순서로 써 주세요.” | 이어폰 준비가 진행자의 할 일로 바뀌었나요? |

짝이 고른 문제 하나만 고쳐 보세요. 활동지에 최초 지시, 바꾼 구절, 실제 답에서 달라진 문장을 남깁니다.

## 예시로 출력 형식을 보여 주고 빈 정보를 처리하기

### 모르는 장소를 어떻게 쓰게 할까요?

공지 C3에는 모임 날짜와 시간이 적혀 있고 장소는 빠져 있습니다. 앞 공지의 201호를 그대로 답하면 다른 원문에서 가져온 사실이 됩니다. “일시와 장소를 알려 줘”를 고쳐서 항목 이름, 장소가 없을 때 쓸 말, 답의 모양을 알려 줍시다. 지시문에는 별개의 예시를 넣고 실제 공지는 input으로 따로 보냅니다.

In [ ]:
no_place = '5월 21일 금요일 오후 4시에 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.'
extract_prompt = """공지에서 모임 일시와 모임 장소를 뽑아 두 줄로 작성하세요. 원문에 없는 항목은 ‘미정’으로 적으세요. 제출 마감과 모임 일시를 구분하세요.
입력 예시: 6월 2일 오후 2시, 별관 101호에서 모입니다.
출력 예시:
모임 일시: 6월 2일 오후 2시
모임 장소: 별관 101호"""
print(ask(extract_prompt, no_place))

모임 일시: 5월 21일 금요일 오후 4시  
모임 장소: 미정


출력 예시에는 별관 101호가 있지만 실제 답의 장소는 ‘미정’입니다. 여기서 ‘미정’은 공지에 장소가 적혀 있지 않다는 표기입니다. 주최자가 아직 장소를 정하지 않았는지는 알 수 없습니다. 예시는 답의 모양을 알려 줍니다. 지금 처리하는 문서의 사실은 input에서 가져와야 합니다. 예시에 쓴 날짜나 장소가 실제 응답에 튀어나온다면 예시와 원문을 더 분명하게 구분하고 장소 없는 예시를 하나 더 제공해 보세요.

### 실습 · 제출 마감도 모임 일시일까요?

아래 공지에는 5월 28일 오전 11시라는 시각이 등장합니다. 무엇을 하기 위한 시각인지 먼저 읽고, 예시가 있는 지시를 새 공지에 보내세요. 모델이 이 값을 모임 일시로 옮겼다면 “무엇의 일시인지”를 직접 설명한 지시로 바꿔 재실행합니다. 짝에게는 다른 입력 예시 하나를 직접 써서 건네세요.

In [ ]:
submission_notice = '5월 28일 금요일 오전 11시까지 지난 모임에서 녹음한 소리를 동아리 게시판에 올려 주세요. 파일 제목에 녹음 장소를 넣어 주세요. 다음 주 모임에서 함께 들을 파일은 진행자가 두 개 고릅니다.'
# extract_prompt와 submission_notice로 답을 받아 두 칸을 원문과 대조하세요.

<details><summary>모범 프롬프트와 응답 살펴보기</summary><pre><code>print(ask(extract_prompt, submission_notice))</code></pre><p>이 지시로 받은 실제 응답</p><pre>모임 일시: 미정  
모임 장소: 미정</pre><p>두 칸이 모두 미정입니다. 오전 11시는 제출 마감이고 다음 모임의 일시와 장소는 적혀 있지 않습니다. 제출 기한을 안내하는 과제라면 먼저 요청하는 항목을 ‘제출 마감’으로 바꿔야 합니다.</p></details>

### 선택 · 결과를 프로그램의 칸으로 받을 때

사람에게 문자로 보낼 때는 위처럼 두 줄의 답을 읽으면 충분합니다. 일정 화면의 날짜 칸과 장소 칸에 각각 값을 넣으려면 두 항목을 정해 둔 구조로 받을 수 있습니다. 아래 Meeting은 답의 칸을 선언합니다. str은 문자열이고, None을 함께 허용한 칸에는 값이 없을 수 있습니다. Pydantic은 이 칸들의 자료형을 확인합니다.

In [ ]:
from pydantic import BaseModel

class Meeting(BaseModel):
    when: str | None
    where: str | None

parsed = client.responses.parse(
    model=model,
    instructions="모임 일시를 when에, 장소를 where에 넣으세요. 공지에 없으면 null로 두세요.",
    input=no_place, text_format=Meeting, store=False,
)
if parsed.output_parsed is not None:
    print(parsed.output_parsed.when)
    print(parsed.output_parsed.where)  # 장소가 없으면 Python에서 None입니다.
else:
    print("구조화된 답을 받지 못했습니다.")

5월 21일 금요일 오후 4시
None


자료형에 맞는 장소 문자열이라도 공지에 없는 101호라면 잘못된 답입니다. 처음에는 날짜와 장소를 원문에서 직접 대조하세요. 다음 수업에서는 상대가 앞에서 말한 날짜를 정정하는 대화를 다룹니다. 지금까지 ask를 부를 때마다 모델이 받은 공지는 그 호출의 input 하나뿐이었습니다.

2026년 11월 26일 Python 3.12, openai 3.35.1로 본문과 접힌 풀이의 대표 요청 8건을 실행했습니다. SDK 참고 문서 · [텍스트 생성](https://developers.openai.com/api/docs/guides/text) · [구조화된 출력](https://developers.openai.com/api/docs/guides/structured-outputs)

### 개념 참고 자료

[Google의 언어 모델 기초 설명](https://developers.google.com/machine-learning/crash-course/llm/transformers) · [트랜스포머를 제안한 논문, Attention Is All You Need](https://arxiv.org/abs/1706.03762)